# Latency test notebook
While the latency testing is built into the data joint pipeline the goal of this notebook is so that the user can quickly inspect the latency of the AR rig offline. To do this you should run at least three sessions for between 5-10 mins each. Then take the PROC files which are outputted by the DLClivegui and move them to their own folder. You can then adjust the fig_save_path and data_path below to that folder and run the notebook to produce the plots. 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import glob

# Set data paths and plotting style params

In [ ]:
figure_save_path = "/Users/thomassainsbury/Documents/code/mathislab/mathis_data/"
data_path = "/Users/thomassainsbury/Documents/code/mathislab/mathis_data/"

In [ ]:

def style():
    """
    Set the style of the plots.

    This function sets the font color, size and weight, and axis label properties
    for a matplotlib plot using the Google style guidelines.
    """
    font_color = "black"
    font_size = 18
    plt.rcParams.update(
        {
            "text.color": font_color,
            "axes.labelcolor": font_color,
            "axes.labelsize": font_size,
            # "axes.titleweight": "bold",
            "axes.titlesize": font_size,
            "xtick.labelcolor": font_color,
            "xtick.labelsize": font_size,
            "ytick.labelcolor": font_color,
            "ytick.labelsize": font_size,
            "font.weight": "regular",
        }
    )

    plt.rc("axes.spines", top=False, bottom=True, left=True, right=False)
    plt.rc("axes", edgecolor=font_color)

style()

# functions

In [ ]:
def filter_pulsed_signal(signal, sample_rate, cutoff_freq=50, filter_order=5):
    """
    Filters high-frequency noise from a pulsed signal and plots the results.
    
    Parameters:
    - signal: The input signal with noise
    - sample_rate: Sampling rate in Hz

    - cutoff_freq: Cutoff frequency for the low-pass filter in Hz (default 50Hz)
    - filter_order: Order of the Butterworth filter (default 5)
    
    Returns:
    - filtered_signal: The filtered output signal
    """
    # Calculate Nyquist frequency
    nyquist = 0.5 * sample_rate
    
    # Design Butterworth low-pass filter
    b, a = butter(filter_order, cutoff_freq/nyquist, btype='low', analog=False)
    
    # Apply zero-phase filtering (filtfilt to avoid phase shift)
    filtered_signal = filtfilt(b, a, signal)
   
    return filtered_signal

def get_signals(dlc, threshold=.2):
    """
    Processes photodiode and generated signal data from a PROC file and returns a merged DataFrame 
    containing aligned signal and photodiode values.

    Args:
        dlc (dict): A dictionary containing photodiode and signal data. It must include the following keys:
            - 'photodiode_read': Array of photodiode readings.
            - 'photodiode_time': Array of photodiode time points.
            - 'start_time': Single start time value to adjust timestamps.
            - 'frame_time': Array of frame time points corresponding to signals.
            - 'time_stamp': Array of signal time stamps.
            - 'signal': Array of signal readings.
    
        std_above_mean (float, optional): Threshold multiplier of standard deviation above the mean 
            for detecting signal changes. Default is 5.

    Returns:
        pd.DataFrame: A DataFrame containing synchronized and processed signal and photodiode data, with the following columns:
            - 'time_stamp': Time stamps 
            - 'send_time': Adjusted signal send times.
            - 'frame_to_socket_time': Absolute difference between the frame and signal send times.
            - 'signal_read': Generated signal values, shifted to frame times 
            - 'photodiode_read': Thresholded photodiode values.
            - 'photodiode_raw_scaled': Scaled raw photodiode readings.

    Notes:
        - The function adjusts the photodiode signal to remove an initial thread-starting point, scales it based on 
          pre-signal mean and max values, and applies a threshold based on the standard deviation of the pre-signal values.
        - The function merges the processed photodiode data with the signal data based on time alignment.
    """
    
    photodiode_read = dlc ["photodiode_read"] 
    photodiode_time = dlc ["photodiode_time"]- dlc ["start_time"]

    # estimate the delay by find the first point where sent signal is 1
    delay = dlc ["frame_time"][np.where(dlc ["signal"] > 0.5)[0][0]]- dlc ["start_time"]
    print("delay: ", delay)
    
    # remove first point as this corresponds to the thread starting
    if len(photodiode_time) != len(photodiode_read): 
        photodiode_read = photodiode_read [1:]
    photodiode_read = photodiode_read [photodiode_time > 1]
    photodiode_time = photodiode_time [photodiode_time > 1]

    flip_photodiode_signal = detect_signal_polarity(photodiode_read)
    filtered_photodiode_read = filter_pulsed_signal(photodiode_read, int(1//np.mean(np.diff(photodiode_time))),cutoff_freq=50) * flip_photodiode_signal
    # Calculate the mean value prior to signal coming in to scale the signal and scale the trace 
    min_start = np.mean(filtered_photodiode_read [(photodiode_time > delay-3) & (photodiode_time < delay-0.5)])
    filtered_photodiode_scaled = ((filtered_photodiode_read-min_start)/(np.max(filtered_photodiode_read)-min_start))
    

    min_start = np.mean(photodiode_read [(photodiode_time > delay-3) & (photodiode_time < delay-0.5)])
    photodiode_read = ((photodiode_read-min_start)/(np.max(photodiode_read)-min_start))
    photodiode_signal_scaled = photodiode_read
    
    # binarise the signal
    photodiode_read = photodiode_read > threshold

    signal_time = dlc ["frame_time"]
    send_time = dlc ["time_stamp"]
    signal_read = dlc ["signal"]
    signal_time = signal_time - dlc ["start_time"]
    send_time = send_time - dlc ["start_time"]
    
    #photo_rising_edge = photodiode_time [np.where((np.diff(photodiode_read,prepend=np.nan)> (std_start*1)) == 1)]
    #signal_rising_edge = signal_time [np.where(np.diff(signal_read,append=np.nan) >0.5)]
    signal = pd.DataFrame({"time_stamp": signal_time, "send_time": send_time, "frame_to_socket_time": abs(signal_time-send_time),  "signal_read": signal_read})
    photodiode = pd.DataFrame({"time_stamp": photodiode_time, "photodiode_read": photodiode_read, "photodiode_raw_scaled": photodiode_signal_scaled, "filtered_photodiode_scaled": filtered_photodiode_scaled, "threshold": threshold})
    df = pd.merge_asof(left=photodiode, right=signal, on="time_stamp")
    
    df ["signal_read"]  = df.signal_read.ffill()
    df ["photodiode_read"]  = df.photodiode_read
   
    return(df)


In [ ]:
def find_rising_edges(time, signal, threshold=0.5):
    "calculates the rising edges of photodiode and signal."
    rising_edges = []
    for i in range(1, len(signal)):
        if signal[i-1] < threshold and signal[i] >= threshold:
            rising_edges.append(time[i])
    return rising_edges


def detect_signal_polarity(photodiode_read):    
    # Calculate the mean value prior to signal coming in to scale the signal and scale the trace 
    read = photodiode_read 
    read = read - np.mean(read [0:100])

    if np.abs(np.min(read)) > np.abs(np.max(read)):
        return -1
        print("flipping signal")
    else:
        print("not flipping signal")
        return 1

    
def get_latency(rising_edges_singal, rising_edges_photodiode):
    latencies = []
    
    # Convert to numpy arrays if they aren't already
    rising_edges_singal = np.array(rising_edges_singal)
    rising_edges_photodiode = np.array(rising_edges_photodiode)
    
    for i in range(len(rising_edges_singal)-1):
        start = rising_edges_singal[i]
        end = rising_edges_singal[i+1]
        
        # Find photodiode edges between current and next signal edge
        mask = (rising_edges_photodiode > start) & (rising_edges_photodiode < end)
        temp_photodiode = rising_edges_photodiode[mask]
        
        if len(temp_photodiode) > 0:
            closest = temp_photodiode[np.argmin(np.abs(temp_photodiode - start))]
            latencies.append({
                'frame_time': start,
                'time_diff': closest - start,
                'photodiode_time': closest
            })
    
    return pd.DataFrame(latencies)

In [ ]:

def get_latencies(path):
    """
    Processes multiple files containing photodiode and signal data to calculate latencies between rising edges of 
    the signal and the photodiode events, and returns a concatenated DataFrame of latency data for all sessions.

    Args:
        path (str): The file path pattern to load the processed data files. The function searches for files ending with 
        "*PROC" in the specified directory.

    Returns:
        pd.DataFrame: A concatenated DataFrame with latency data from all processed sessions, containing the following columns:
            - 'time_diff': The time difference between the rising edge of the signal and the rising edge of the photodiode event.
            - 'session': The index of the session (file) from which the latencies are computed.
            - 'frame_to_socket': The average difference between time stamps and frame times for the given session.

    Notes:
        - The function loads each processed data file (ending with "*PROC"), extracts the photodiode and signal data using the 
          `get_signals` function, and calculates latencies between the rising edges of the signal and photodiode readings.
        - For each session, it computes the average 'frame_to_socket' time difference, which is the difference between 
          the signal's send times and frame times, averaged over a portion of the data.
        - The results for all sessions are concatenated into a single DataFrame.
    """
    
    
    files = glob.glob(path + "*PROC")
    data_list = []
    rate_list = []
    for i, f in enumerate(files):
        print(i, " ", f)
        dlc = np.load(f, allow_pickle=True)
        df = get_signals(dlc, threshold=0.5)
        
        # get rising edges of the binarised signals
        rising_edges_a = find_rising_edges(df.time_stamp, df.signal_read)
        rising_edges_photodiode = find_rising_edges(df.time_stamp, df.photodiode_read)
        latencies = get_latency(rising_edges_a, rising_edges_photodiode)
        
        latencies ["session"] =  i
        latencies ["frame_to_socket"] = np.mean(dlc ["time_stamp"][100:]- dlc ["frame_time"][100:] )
        
        data_list.append(latencies)

    return(pd.concat(data_list))



In [ ]:
latencies = get_latencies(data_path)
mean_latencies = latencies.groupby(["session"]).mean()
print(mean_latencies.time_diff)

In [ ]:
g = sns.kdeplot(data=latencies, x= latencies.time_diff*1000, hue="session", palette=ListedColormap(['black']), alpha=0.5, legend=False)
plt.xlim(0,125)
plt.xlabel("Latency (ms)")
plt.axvline(np.mean(latencies.time_diff*1000), c="red", linestyle="dashed", alpha = 0.7)
print(np.mean(latencies.time_diff))
print(np.std(latencies.time_diff))
sns.despine(offset=10)
plt.savefig(figure_save_path + "roundtrip_latency.svg")

In [ ]:
plt.figure(figsize=(4,5))

sns.violinplot(x=np.repeat("DLClive", len(mean_latencies ["time_diff"])), y=abs(mean_latencies ["frame_to_socket"])*1000, color="cornflowerblue")
sns.violinplot(x=np.repeat("Unity", len(mean_latencies ["time_diff"])), y=(abs(mean_latencies ["time_diff"])*1000) - (abs(mean_latencies ["frame_to_socket"])*1000), color="purple")
sns.violinplot(x=np.repeat("Total", len(mean_latencies ["time_diff"])), y=abs(mean_latencies ["time_diff"])*1000, color="black", alpha=0.7)

plt.ylabel("Latency (ms)")
plt.ylim(0,120)
sns.despine(offset=10)
plt.savefig(figure_save_path + "latencies.svg")

In [ ]:
# print DLC processing steps latency
print(np.mean(abs(mean_latencies ["frame_to_socket"])*1000))
print(np.std(abs(mean_latencies ["frame_to_socket"])*1000))

In [ ]:
# Unity render latency
print(np.mean((mean_latencies ["time_diff"]*1000) - (mean_latencies ["frame_to_socket"])*1000))
print(np.std((mean_latencies ["time_diff"]*1000) - (mean_latencies ["frame_to_socket"])*1000))

In [ ]:
# print total latency
print(np.mean((mean_latencies ["time_diff"])*1000))
print(np.std((mean_latencies ["time_diff"])*1000))

In [ ]:
# plot raw data from a single session
files = glob.glob(data_path + "*PROC")
dlc = np.load(files[1], allow_pickle=True)
df = get_signals(dlc, threshold=0.5)
rising_edges_signal = find_rising_edges(df.time_stamp, df.signal_read)
rising_edges_photodiode = find_rising_edges(df.time_stamp, df.photodiode_read)
latencies = get_latency(rising_edges_signal, rising_edges_photodiode)

start_time = df.time_stamp[np.where(df.signal_read == 1)[0][0]]
print("start_time", start_time)

fig, ax = plt.subplots(3,1,figsize=(10,5))
ax[0].plot(df.time_stamp, df.signal_read,c="black",linewidth=2)


ax[1].plot(df.time_stamp, df.photodiode_raw_scaled,c="#AA1BCE",linewidth=2)

ax[2].plot(df.time_stamp,  df.signal_read, c="black",linewidth=2,alpha=0.8)
ax[2].plot(df.time_stamp,  df.photodiode_read, c="#AA1BCE",linewidth=2,alpha=0.8)

for ft, td in zip(latencies.frame_time, latencies.time_diff):
    ax[2].annotate(round(td*1000, 2), (ft, 1.1))

ax[1].set_ylim(-0.2,1.3)

ax[1].axhline(df.threshold[0],c="red",linestyle="dashed",alpha=0.5, linewidth=3)
ax[2].set_ylim(-0.2,1.3)


sns.despine(offset=10)

for a in ax:
    a.axis('off')
    a.set_xlim(start_time-1,start_time+1)

plt.savefig(figure_save_path + "signal.svg")

In [ ]:
main_data_path = "/Users/thomassainsbury/Documents/code/mathislab/"

In [ ]:
all_latencies = []
for d in ["/Yangs_data/", "/mathis_data/"]:

    latencies = get_latencies(main_data_path + d)
    mean_latencies = latencies.groupby(["session"]).mean()
    mean_latencies ["lab"] = d
    all_latencies.append(mean_latencies)
all_latencies = pd.concat(all_latencies)



In [ ]:
plt.figure(figsize=(3,5))
sns.violinplot(x="lab", y=all_latencies.time_diff*1000, data=all_latencies, color="black",alpha=0.6)
plt.ylim(0,120)
plt.xticks([0,1], ["Tolias", "Mathis"])
plt.ylabel("Latency (ms)")
plt.xlabel("Lab")
sns.despine(offset=10)
plt.savefig(figure_save_path + "latencies_both_labs.svg")


